<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_27_practicum_dp_greedy/re_lesson_27_greedy_regex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔎 Урок 27 (додатково) — пошук у тексті: модуль `re` і «жадібність»

Слово «жадібний» у курсі трапляється двічі: **жадібний алгоритм** (урок 27) і **жадібний квантифікатор** у регулярних виразах. Назва спільна, поведінка — різна. Цей ноутбук веде від простого пошуку в рядку до регулярних виразів на журналі водіїв «Смачно + Таксі».

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 1. Методи рядків | коли `in`, `find`, `replace` досить, а коли — ні | 1.1–1.2 | |
| 2. Перший шаблон | `re.search`, об'єкт `Match`, `None`, сирі рядки `r"..."` | 2.1–2.3 | |
| 3. Будівельні блоки | класи символів, набори, квантифікатори, межі, групи, іменовані групи, `\|` | 3.1–3.8 | |
| 4. Функції `re` | `match` / `search` / `fullmatch`, `findall`, `finditer`, `sub`, `split`, `compile` і прапорці | 4.1–4.8 | **Задача 1** |
| 5. Жадібні й нежадібні | `.*` проти `.*?`, backtracking, `[^…]`, пастки `+?`, катастрофічне відступання | 5.1–5.6 | **Задача 2** |
| 6. Межі regex | чому не для HTML; `html.parser` | 6.1–6.2 | |
| 7. Журнал диспетчерської | звіт по журналу, рядки поїздок | 7.1 | **Задача 3** |

Базова таблиця функцій `re` уже була в уроці 13 (`notes_modules.ipynb`, частина 11). Тут — детальніше й з пастками.

**Як працювати**

- Виконуй клітинки **зверху вниз**.
- Клітинка з визначенням (`def`, дані) лише створює їх — **запуск у наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Книга: [Урок 27, розділ «жадібність» у регулярних виразах](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_27/).

---
## 1. Спершу — методи рядків

### Приклад 1.1. Фіксований текст — без regex

Якщо шукаєш **конкретний фіксований текст**, регулярні вирази не потрібні: методи рядків простіші, читабельніші й швидші.

In [1]:
text = "Курс Python розробки"

print("Python" in text)                    # чи є
print(text.find("Python"))                 # де (або -1)
print(text.replace("Python", "Python 3"))  # замінити
print(text.startswith("Курс"))

True
5
Курс Python 3 розробки
True


`True` — підрядок є; `5` — його позиція (рахуємо з нуля); `replace` повертає **новий** рядок; `startswith` перевіряє початок.

### Приклад 1.2. Коли методів рядків замало

Задача: дістати з повідомлення всі номери замовлень. Спробуймо лише методами рядків — розбити на слова й залишити ті, що складаються з цифр:

In [2]:
message = "Замовлення #10452 підтверджено. Попередні: #892 та Order-771, сума 250."

In [5]:
print([word for word in message.split()])

['Замовлення', '#10452', 'підтверджено.', 'Попередні:', '#892', 'та', 'Order-771,', 'сума', '250.']


In [3]:
print([word for word in message.split() if word.isdigit()])

[]


In [6]:
print([word.strip("#.,") for word in message.split() if word.strip("#.,").isdigit()])

['10452', '892', '250']


Перша спроба не знайшла нічого: у кожному «слові» з цифрами є ще щось — `#`, дефіс, кома чи крапка (`250.`). Друга, зі `strip`, уже бере `10452` і `892`, але досі губить `Order-771` і помилково бере суму `250`. Кожне нове правило — ще одна латка. Саме тут потрібні регулярні вирази: вони описують **шаблон** тексту — «`#` або `Order-`, потім цифри».

---
## 2. Перший шаблон: `re.search`

### Приклад 2.1. Об'єкт `Match`

`re.search(шаблон, текст)` шукає **перший** збіг будь-де в тексті. Повертає об'єкт `Match` або `None`. `\d` означає «будь-яка цифра», `{2}` — «рівно двічі», дужки `( )` — **групи**, частини збігу, які можна дістати окремо.

In [7]:
import re

In [8]:
m = re.search(r"(\d{2}):(\d{2})", "Старт зміни о 09:15")

In [9]:
print(m)

<re.Match object; span=(14, 19), match='09:15'>


In [10]:
print(m.group())      # увесь збіг

09:15


In [11]:
print(m.group(1))     # перша група в дужках

09


In [12]:
print(m.groups())     # усі групи
print(m.start())      # позиція початку

('09', '15')
14


`Match` знає все про збіг: `group()` — увесь текст `09:15`, `group(1)` — перша група `09`, `groups()` — усі групи кортежем, `start()` — позиція 14, з якої починається збіг.

### Приклад 2.2. Немає збігу — `None`

Якщо збігу немає, `re.search` повертає `None`, і `.group()` на ньому впаде з `AttributeError`. Тому результат спершу перевіряють.

In [13]:

def shift_start(text):
    """Час «ГГ:ХХ» з тексту або None, якщо часу немає."""
    m = re.search(r"\d{2}:\d{2}", text)
    if m:                         # Match — істинний, None — хибний
        return m.group()
    return None

In [15]:
print(shift_start("Старт зміни о 09:15"))
print(shift_start("Старт зміни зранку"))

09:15
None


In [16]:
try:
    re.search(r"\d{2}:\d{2}", "зранку").group()
except AttributeError as error:
    print("AttributeError:", error)

AttributeError: 'NoneType' object has no attribute 'group'


`09:15`, потім `None`, а прямий виклик `.group()` на `None` дає `'NoneType' object has no attribute 'group'`. **Правило:** `if m:` перед `m.group()`.

### Приклад 2.3. Сирі рядки `r"..."`

У звичайному рядку Python сам обробляє `\`: `"\n"` — перенос, `"\b"` — службовий символ backspace. А в regex `\b` означає **межу слова**. Щоб Python не чіпав зворотні скісні, шаблони пишуть **сирими рядками** з префіксом `r`.

**🔮 Прогноз:** що виведуть обидва рядки?

```python
print(re.search("\bOrder", "Order-771"))
print(re.search(r"\bOrder", "Order-771"))
```

<details>
<summary>Відповідь</summary>

`None`, потім `<re.Match object; span=(0, 5), match='Order'>`. Без `r` Python перетворив `"\b"` на символ backspace (`'\x08'`), і regex шукав його — а його в тексті немає.

</details>

In [17]:
print(re.search("\bOrder", "Order-771"))

None


In [18]:
print(re.search(r"\bOrder", "Order-771"))

<re.Match object; span=(0, 5), match='Order'>


In [19]:
print(repr("\b"))

'\x08'


**Правило:** шаблон regex — завжди `r"..."`.

---
## 3. Будівельні блоки

---
## 3. Будівельні блоки

| Шаблон | Значення | Приклад |
|---|---|---|
| `\d` / `\D` | цифра / не цифра | `\d+` — `771` |
| `\w` / `\W` | літера, цифра або `_` / інше | `\w+` — `Хрещатик` |
| `\s` / `\S` | пробільний символ (пробіл, `\t`, `\n`) / інше | `\s+` |
| `.` | будь-який символ, крім переносу рядка | |
| `[...]` | один символ з набору | `[АБВ]`, `[0-9]` |
| `^` / `$` | початок / кінець рядка | |
| `\b` | межа слова | `\bOrder` |

**Квантифікатори** — скільки разів повторити попереднє:

| Квантифікатор | Скільки |
|---|---|
| `*` | 0 або більше |
| `+` | 1 або більше |
| `?` | 0 або 1 (необов'язкове) |
| `{n}` / `{n,m}` | рівно `n` / від `n` до `m` |

**Групи:** `(...)` — запам'ятати частину збігу; `(?:...)` — лише згрупувати, не запам'ятовуючи; `a|b` — «або».

### Приклад 3.1. Класи символів `\d`, `\w`, `\s`

Один і той самий рядок — три класи. `findall` повертає всі збіги списком; `+` — «один або більше», щоб брати цілі шматки, а не по символу.

In [20]:
sample = "Кур'єр D-17 о 09:15\tприйняв #104 (Хрещатик)"

In [21]:
print(re.findall(r"\d+", sample))

['17', '09', '15', '104']


In [22]:
print(re.findall(r"\w+", sample))

['Кур', 'єр', 'D', '17', 'о', '09', '15', 'прийняв', '104', 'Хрещатик']


In [23]:
print(re.findall(r"\s", sample))

[' ', ' ', ' ', '\t', ' ', ' ']


In [24]:
print(re.findall(r"\S+", sample))

["Кур'єр", 'D-17', 'о', '09:15', 'прийняв', '#104', '(Хрещатик)']


- `\d+` — шматки з цифр: `17`, `09`, `15`, `104`;
- `\w+` — «слова»: літери (і українські теж — у Python 3 `\w` розуміє Unicode), цифри, `_`. Апостроф і дефіс до `\w` не входять, тому `Кур'єр` розпалось на `Кур` і `єр`;
- `\s` — пробільні символи: пробіли й табуляція `\t`;
- `\S+` — усе між пробілами, разом з дужками й `#`.

### Приклад 3.2. Набори `[...]` і заперечення `[^...]`

`[...]` — **один** символ з набору; діапазон пишуть через дефіс. `[^...]` — будь-який символ, **крім** перелічених.

In [25]:
codes = "Зони: A1, b2, Г7, Ї9, z0, 55"

In [26]:
print(re.findall(r"[A-Za-z]\d", codes))         # латинська літера + цифра

['A1', 'b2', 'z0']


In [27]:
print(re.findall(r"[А-ЯҐЄІЇа-яґєії]\d", codes))     # українська літера + цифра

['Г7', 'Ї9']


In [28]:
print(re.findall(r"[^\s,:]+", codes))               # усе, крім пробілів, ком і двокрапок

['Зони', 'A1', 'b2', 'Г7', 'Ї9', 'z0', '55']


Латинський діапазон дав `A1`, `b2`, `z0`; український — `Г7`, `Ї9`. Літери `Ґ`, `Є`, `І`, `Ї` у таблиці Unicode стоять **поза** діапазоном `А-Я`, тому їх дописують окремо — без цього `Ї9` загубилось би. Останній рядок — «слова» між розділювачами: заперечений набір часто точніший за `.`.

### Приклад 3.3. Квантифікатори `?`, `+`, `{n,m}`

`?` — необов'язкове (0 або 1 раз), `{n,m}` — від `n` до `m` разів. Шаблон нижче шукає суму: 1–5 цифр, необов'язковий пробіл, `грн` і необов'язкова крапка.

In [33]:
amounts = "250 грн, 1200грн, 80 грн., 999999 грн, 0 грн,   грн"
phones = "+380501234567, 0501234567, +38050123"

In [45]:
print(re.findall(r"\b\d{1,5} ?грн\.?", amounts))

['250 грн', '1200грн', '80 грн.', '0 грн']


In [36]:
print(re.findall(r"(?:\+38)?0\d{9}\b", phones))

['+380501234567', '0501234567']


Суми — `250 грн`, `1200грн`, `80 грн.`; шість цифр `999999` у шаблон `\d{1,5}` не вміщаються, а `\b` не дає взяти «хвіст» `99999`; голе `грн` без числа не підходить. Телефони — повний `+380501234567` і короткий `0501234567`: `(?:\+38)?` — необов'язковий код країни (`+` екрануємо, бо це квантифікатор). Обрізаний `+38050123` не дав збігу.

### Приклад 3.4. Межі: `^`, `$`, `\b`

`^` і `$` — початок і кінець рядка, `\b` — межа між «словом» (`\w`) і не-словом. Вони не з'їдають символів — лише перевіряють **позицію**.

In [46]:
words = ["Order-771", "PreOrder-5", "Order", "Order-12 скасовано"]

In [50]:
for w in words:
    print(re.search(r'\bOrder', w))

<re.Match object; span=(0, 5), match='Order'>
None
<re.Match object; span=(0, 5), match='Order'>
<re.Match object; span=(0, 5), match='Order'>


In [47]:
for w in words:
    print(f"{w!r:22} \\bOrder: {bool(re.search(r'\bOrder', w))!s:5}  "
          f"^Order-\\d+$: {bool(re.search(r'^Order-\d+$', w))}")

'Order-771'            \bOrder: True   ^Order-\d+$: True
'PreOrder-5'           \bOrder: False  ^Order-\d+$: False
'Order'                \bOrder: True   ^Order-\d+$: False
'Order-12 скасовано'   \bOrder: True   ^Order-\d+$: False


`\bOrder` не знайшов `PreOrder-5` (перед `O` стоїть літера `e` — межі слова немає), а `^Order-\d+$` пропустив лише рядок, що **цілком** має вигляд `Order-цифри`: `Order-12 скасовано` має зайвий хвіст.

### Приклад 3.5. Звичайна група проти `(?:...)`

In [54]:
text = "Замовлення #10452 підтверджено. Попередні: #892 та Order-771."

In [52]:
print(re.findall(r"(?:#|Order-)\d+", text))   # група без запам'ятовування

['#10452', '#892', 'Order-771']


In [53]:
print(re.findall(r"(#|Order-)\d+", text))     # звичайна група

['#', '#', 'Order-']


Якщо в шаблоні є **звичайна** група, `findall` повертає вміст групи, а не весь збіг — звідси `['#', '#', 'Order-']`. Коли дужки потрібні лише для `|`, пиши `(?:...)`.

### Приклад 3.6. Іменовані групи `(?P<ім'я>...)`

Номери груп `1`, `2`, `3` легко переплутати. Групі можна дати ім'я, а `groupdict()` поверне словник.

In [55]:
TRIP = r"(?P<time>\d{2}:\d{2}) (?P<courier>D-\d+) (?P<order>\d+) (?P<kind>picked|delivered)"

In [56]:
m = re.search(TRIP, "18:03 D-1 101 picked")

In [57]:
print(m.group("courier"), m["order"])

D-1 101


In [58]:
print(m.groupdict())

{'time': '18:03', 'courier': 'D-1', 'order': '101', 'kind': 'picked'}


`m.group("courier")` і `m["order"]` — те саме, що номери груп, але читається без лічби. `groupdict()` дає готовий словник — зручно для розбору рядків журналу.

### Приклад 3.7. Пріоритет `|`

`|` має **найнижчий** пріоритет: `^#|Order-\d+` означає «(`^#`) або (`Order-\d+`)», а не «`#` чи `Order-` на початку, потім цифри». Дужки змінюють це.

In [59]:
for text in ["#104", "Order-771", "#", "звіт Order-5"]:
    loose = re.search(r"^#|Order-\d+", text)
    strict = re.search(r"^(?:#|Order-)\d+", text)
    print(f"{text!r:14} без дужок: {loose and loose.group()!r:12} з дужками: {strict and strict.group()!r}")

'#104'         без дужок: '#'          з дужками: '#104'
'Order-771'    без дужок: 'Order-771'  з дужками: 'Order-771'
'#'            без дужок: '#'          з дужками: None
'звіт Order-5' без дужок: 'Order-5'    з дужками: None


Без дужок голий `#` збігся (першій альтернативі цифри не потрібні), а `#104` дав лише `#`; `Order-5` знайдено навіть посеред рядка, бо `^` належить лише першій альтернативі. З дужками `(?:#|Order-)\d+` усе працює як задумано. (`a and a.group()` — коротко «`None`, якщо збігу немає».)

### Приклад 3.8. Номери замовлень

Зводимо: номер — `#` або `Order-`, потім одна чи більше цифр. Група без запам'ятовування — щоб `findall` повернув збіг цілком.

In [60]:
journal = """Замовлення #10452 підтверджено. Попередні: #892 та Order-771.
Зустрічі: 15.04.2024, 31.02.2024 (помилка) та 09.05.2024.
Коментар  водія:\tклієнт   просив   "без цибулі"  і "дзвонити в двері"."""


def order_numbers(text):
    return re.findall(r"(?:#|Order-)\d+", text)

In [61]:
print(journal)

Замовлення #10452 підтверджено. Попередні: #892 та Order-771.
Зустрічі: 15.04.2024, 31.02.2024 (помилка) та 09.05.2024.
Коментар  водія:	клієнт   просив   "без цибулі"  і "дзвонити в двері".


In [62]:

print(order_numbers(journal))
print(order_numbers(message), order_numbers("без номерів"))
assert order_numbers(journal) == ["#10452", "#892", "Order-771"]

['#10452', '#892', 'Order-771']
['#10452', '#892', 'Order-771'] []


Журнал диспетчерської за день — три рядки з номерами, датами й коментарем. `order_numbers` знайшов усі три номери, а на повідомленні з прикладу 1.2 — також без суми `250`, яку плутали методи рядків. Немає номерів — порожній список.

---
## 4. Функції модуля `re`

---
## 4. Функції модуля `re`

| Функція | Що робить | Повертає |
|---|---|---|
| `re.search(p, s)` | перший збіг будь-де | `Match` або `None` |
| `re.match(p, s)` | збіг лише **на початку** рядка | `Match` або `None` |
| `re.fullmatch(p, s)` | **весь** рядок відповідає шаблону | `Match` або `None` |
| `re.findall(p, s)` | усі збіги | список рядків (або кортежів, якщо груп кілька) |
| `re.finditer(p, s)` | усі збіги по одному | ітератор `Match` (урок 11) |
| `re.sub(p, repl, s)` | замінити всі збіги | новий рядок |

### Приклад 4.1. `match`, `search`, `fullmatch`

In [63]:
print(re.match(r"\d+", "Order-771 на 250"))    # на початку — літери

None


In [64]:
print(re.search(r"\d+", "Order-771 на 250"))   # будь-де

<re.Match object; span=(6, 9), match='771'>


In [65]:
print(re.fullmatch(r"\d{2}:\d{2}", "09:15"))

<re.Match object; span=(0, 5), match='09:15'>


In [66]:
print(re.fullmatch(r"\d{2}:\d{2}", "09:15 ранку"))

None


`match` шукає лише з **початку** рядка — там літери, тому `None`; `search` знаходить `771` всередині; `fullmatch` вимагає, щоб шаблону відповідав **увесь** рядок: `09:15` — так, `09:15 ранку` — ні.

### Приклад 4.2. Перевірка формату: `fullmatch` + точні діапазони

`\d{2}:\d{2}` пропустить і `99:99`. Точніше: година `[01]\d|2[0-3]` (00–19 або 20–23), хвилини `[0-5]\d`. Для **перевірки** введення — завжди `fullmatch`, інакше `search` знайде «щось схоже» всередині.

In [67]:
TIME = r"(?:[01]\d|2[0-3]):[0-5]\d"


def is_time(text):
    return re.fullmatch(TIME, text) is not None

In [69]:
for t in ["09:15", "23:59", "24:00", "9:15", "12:60", "09:15 ", "у 09:15", "99:15"]:
    print(f"{t!r:10} fullmatch: {is_time(t)!s:5}  search: {re.search(TIME, t) is not None}")

'09:15'    fullmatch: True   search: True
'23:59'    fullmatch: True   search: True
'24:00'    fullmatch: False  search: False
'9:15'     fullmatch: False  search: False
'12:60'    fullmatch: False  search: False
'09:15 '   fullmatch: False  search: True
'у 09:15'  fullmatch: False  search: True
'99:15'    fullmatch: False  search: False


`fullmatch` пропустив лише `09:15` і `23:59`. `search` дав `True` ще й для `"09:15 "` і `"у 09:15"` — знайшов час **усередині**. Це правильно для пошуку, але не для перевірки форми.

### Приклад 4.3. Що повертає `findall`: залежить від груп

Той самий текст — три шаблони: без груп, з однією групою, з двома.

In [70]:
pays = "09:15 250 грн; 10:02 180 грн; 10:40 320 грн"

In [71]:
print(re.findall(r"\d+ грн", pays))                 # без груп — увесь збіг

['250 грн', '180 грн', '320 грн']


In [72]:
print(re.findall(r"(\d+) грн", pays))               # одна група — її вміст

['250', '180', '320']


In [73]:
print(re.findall(r"(\d{2}:\d{2}) (\d+) грн", pays)) # дві групи — кортежі

[('09:15', '250'), ('10:02', '180'), ('10:40', '320')]


Без груп — рядки цілком (`'250 грн'`); з однією групою — лише вміст групи (`'250'`); з двома — список кортежів (`('09:15', '250')`). Числа лишаються **рядками** — `int()` треба викликати самому.

### Приклад 4.4. `finditer` і `re.M`

Журнал водія — по рядку на замовлення. `finditer` віддає збіги **по одному** (ітератор, урок 11) — разом з позицією `span()`. Прапорець `re.M` (multiline) змушує `^` і `$` працювати для **кожного рядка**, а не лише для всього тексту.

In [74]:
log = """09:15 #104 Хрещатик 250 грн
10:02 #105 Поділ 180 грн
10:40 Order-106 Оболонь 320 грн"""

In [75]:
for m in re.finditer(r"(\d+) грн", log):
    print(m.group(1), m.span())


250 (20, 27)
180 (45, 52)
320 (77, 84)


In [76]:
print(re.findall(r"^\d{2}:\d{2}", log))          # ^ — лише початок усього тексту


['09:15']


In [77]:
print(re.findall(r"^\d{2}:\d{2}", log, re.M))    # ^ — початок кожного рядка


['09:15', '10:02', '10:40']


Три суми з позиціями в тексті. Без `re.M` `^` бачить лише початок усього тексту — один час `09:15`; з `re.M` — по часу з кожного рядка.

### Приклад 4.5. `sub`: заміна з групами і функцією

`re.sub(шаблон, заміна, текст)` замінює **всі** збіги. У заміні `\1`, `\2` — вміст груп. Замість рядка можна передати **функцію**: вона отримує `Match` і повертає текст заміни.

In [78]:
def mask_phone(m):
    """+380501234567 → +38050*****67: лишаємо код оператора й дві останні цифри."""
    digits = m.group()
    return digits[:6] + "*" * (len(digits) - 8) + digits[-2:]

In [79]:
print(re.sub(r"(\d{2})\.(\d{2})\.(\d{4})", r"\3-\2-\1", "15.04.2024 і 09.05.2024"))

2024-04-15 і 2024-05-09


In [80]:
print(re.sub(r"\+380\d{9}", mask_phone, "Клієнт +380501234567, кур'єр +380671112233"))


Клієнт +38050*****67, кур'єр +38067*****33


In [81]:
print(re.subn(r"#(\d+)", r"№\1", "#104, #105, #106"))

('№104, №105, №106', 3)


Дати переписано у формат `РРРР-ММ-ДД` (`\3` — рік, `\1` — день). Телефони замасковано функцією — по-різному для кожного збігу. `re.subn` повертає ще й **кількість** замін: `('№104, №105, №106', 3)`.

### Приклад 4.6. `re.split`: кілька розділювачів

`str.split` вміє лише один розділювач. `re.split` ділить за шаблоном — наприклад, кома **або** крапка з комою, з будь-якими пробілами після.

In [83]:
wishes = "без цибулі;  дзвонити в двері, решта   — без здачі;не стукати"

In [84]:
print(wishes.split(","))

['без цибулі;  дзвонити в двері', ' решта   — без здачі;не стукати']


In [85]:
print(re.split(r"\s*[;,]\s*", wishes))

['без цибулі', 'дзвонити в двері', 'решта   — без здачі', 'не стукати']


`str.split(",")` розділив лише по комі й лишив `;` та пробіли всередині. `re.split(r"\s*[;,]\s*")` дав чотири чисті побажання.

### Приклад 4.7. `re.compile` і прапорець `re.I`

Шаблон, який використовують багато разів, компілюють один раз: `re.compile` повертає об'єкт з тими самими методами (`search`, `findall`, `sub`…). Прапорці передають другим аргументом; `re.I` ігнорує регістр.

In [89]:
STATUS = re.compile(r"\b(скасовано|доставлено)\b", re.I)
updates = ["#104 Доставлено", "#105 СКАСОВАНО клієнтом", "#106 в дорозі"]

In [88]:
STATUS.findall("#105 СКАСОВАНО клієнтом")

['СКАСОВАНО']

In [90]:
for line in updates:
    m = STATUS.search(line)
    print(f"{line:26} → {m.group(1).lower() if m else '—'}")
print(STATUS.pattern, STATUS.flags & re.I != 0)

#104 Доставлено            → доставлено
#105 СКАСОВАНО клієнтом    → скасовано
#106 в дорозі              → —
\b(скасовано|доставлено)\b True


`re.I` знайшов і `Доставлено`, і `СКАСОВАНО`; рядок без статусу дав `—`. Скомпільований шаблон пам'ятає свій текст (`.pattern`) і прапорці. (Python і сам кешує нещодавні шаблони, тому `compile` — передусім про читабельність: шаблон має ім'я.)

### Приклад 4.8. Прибрати зайві пробіли

У коментарі водія — подвійні пробіли, табуляції й переноси. `\s+` збігається з **будь-якою** послідовністю пробільних символів; заміна на один пробіл і `strip()` по краях дають акуратний рядок.

In [91]:
def squeeze_spaces(text):
    return re.sub(r"\s+", " ", text).strip()

In [92]:
print(repr(squeeze_spaces("Python   і   модуль\tre  --  це \t \t потужно! ")))
print(repr(squeeze_spaces("  а\n\nб  ")))
print(squeeze_spaces(journal.splitlines()[2]))
assert squeeze_spaces("Python   і   модуль\tre  --  це \t \t потужно! ") == "Python і модуль re -- це потужно!"

'Python і модуль re -- це потужно!'
'а б'
Коментар водія: клієнт просив "без цибулі" і "дзвонити в двері".


Усі серії пробілів, `\t` і `\n\n` стали одним пробілом, краї обрізано. Третій рядок журналу теж став читабельним.

### Задача 1. Дати: формат і календар

Regex перевіряє **формат** `DD.MM.YYYY`, але не **календар**: `31.02.2024` має правильний вигляд, хоча такої дати немає. Напиши `valid_dates(text)`: знайди всі дати за шаблоном (дві цифри, крапка, дві цифри, крапка, чотири цифри; по краях — `\b`), а потім залиш лише справжні, перевіривши кожну через `datetime.strptime(рядок, "%d.%m.%Y")` (він кидає `ValueError` для неіснуючої дати — урок 14).

Поверни список об'єктів `date` (`.date()` від результату `strptime`). Перевірка — у наступній клітинці.

In [93]:
from datetime import date, datetime


def valid_dates(text):
    # BEGIN SOLUTION
    result = []
    for raw in re.findall(r"\b\d{2}\.\d{2}\.\d{4}\b", text):
        try:
            result.append(datetime.strptime(raw, "%d.%m.%Y").date())
        except ValueError:
            print("Календарно неіснуюча дата:", raw)
    return result
    # END SOLUTION

In [94]:
print(valid_dates(journal))
assert valid_dates(journal) == [date(2024, 4, 15), date(2024, 5, 9)]
assert valid_dates("29.02.2024 і 29.02.2023") == [date(2024, 2, 29)]
assert valid_dates("версія 1.02.2024.1 і 123.04.2024") == []
print("✅ Задача 1 розв'язана")

Календарно неіснуюча дата: 31.02.2024
[datetime.date(2024, 4, 15), datetime.date(2024, 5, 9)]
Календарно неіснуюча дата: 31.02.2024
Календарно неіснуюча дата: 29.02.2023
✅ Задача 1 розв'язана


---
## 5. Жадібні й нежадібні квантифікатори

### Приклад 5.1. `.*` проти `.*?`

**🔮 Прогноз:** що виведе кожен рядок?

```python
html = "<b>Python</b><i>курс</i>"
print(re.findall(r"<.*>", html))
print(re.findall(r"<.*?>", html))
```

<details>
<summary>Відповідь</summary>

`['<b>Python</b><i>курс</i>']` і `['<b>', '</b>', '<i>', '</i>']`.

</details>

In [95]:
html = "<b>Python</b><i>курс</i>"
print(re.findall(r"<.*>", html))
print(re.findall(r"<.*?>", html))

['<b>Python</b><i>курс</i>']
['<b>', '</b>', '<i>', '</i>']


**Жадібний** `.*` (за замовчуванням `*` і `+` жадібні):

1. `<` збігається з першою дужкою;
2. `.*` з'їдає **весь** залишок рядка;
3. шаблону ще потрібен `>`, а текст скінчився — рушій **відступає** (backtracking) на один символ назад, потім ще на один… доки не натрапить на `>`. Перший знайдений з кінця `>` — останній у рядку.

**Нежадібний** `.*?` (знак `?` після квантифікатора): бере **якомога менше** і після кожного символу перевіряє, чи не можна вже закінчити. Перший же `>` закриває збіг.


### Приклад 5.2. Дужки в журналі

Те саме з поясненнями в дужках у рядку журналу: жадібний квантифікатор «склеює» дві пари дужок.

In [96]:
note = "#104 (без цибулі) прийнято, #105 (дзвонити) в дорозі"

In [97]:
print(re.findall(r"\((.*)\)", note))
print(re.findall(r"\((.*?)\)", note))

['без цибулі) прийнято, #105 (дзвонити']
['без цибулі', 'дзвонити']


`\(` — екранована дужка (без `\` дужка означала б групу). Жадібний `(.*)` захопив усе від першої `(` до **останньої** `)`, разом з `прийнято, #105 (`; нежадібний `(.*?)` дав два окремі пояснення.

### Приклад 5.3. Заперечений набір `[^…]` — точніше за `.*?`

Часто ще краще сказати прямо, **що не може** стояти всередині: «будь-що, крім закривної дужки». Такий шаблон не відступає взагалі — він фізично не може «перескочити» `]`.

In [98]:
tags = "[терміново] #104 [VIP] [оплачено карткою]"

In [99]:
print(re.findall(r"\[(.*)\]", tags))
print(re.findall(r"\[(.*?)\]", tags))
print(re.findall(r"\[([^\]]*)\]", tags))

['терміново] #104 [VIP] [оплачено карткою']
['терміново', 'VIP', 'оплачено карткою']
['терміново', 'VIP', 'оплачено карткою']


Жадібний — один довгий шматок; нежадібний і `[^\]]*` — три мітки. Для простих розділювачів `[^…]*` — найнадійніший вибір: його поведінка не залежить від того, що стоїть далі в шаблоні.

### Приклад 5.4. Пастка: нежадібний бере **мінімум**

`+?` — «один або більше, але якомога менше». Без того, що стоїть після нього в шаблоні, цей мінімум — **один** символ.

In [100]:
print(re.findall(r"\d+", "771 і 50"))
print(re.findall(r"\d+?", "771 і 50"))
print(re.search(r"#(\d+?)\b", "#10452").group(1))

['771', '50']
['7', '7', '1', '5', '0']
10452


`\d+?` порізав числа по одній цифрі: `['7', '7', '1', '5', '0']`. А з `\b` після групи нежадібний мусив дійти до межі слова — і взяв усе `10452`. **Правило:** нежадібний квантифікатор має сенс, лише коли за ним іде «стоп-сигнал» (`>`, `"`, `\b`…).

### Приклад 5.5. Скільки кроків «відступає» `.*`

Рушій regex не друкує своїх кроків, але їх можна порахувати вручну для `<.*>` на рядку з прикладу 5.1. Функція нижче імітує жадібний `.*`: спершу бере все до кінця, а потім відступає по символу, доки наступний символ не буде `>`.

In [ ]:
def greedy_star_then(text, start, stop):
    """Імітація `.*` + stop від позиції start: повертає (кінець збігу, кількість відступів)."""
    end = len(text)                  # .* з'їв усе до кінця
    backtracks = 0
    while end > start:
        if text[end - 1] == stop:     # чи можна закінчити на цьому символі
            return end, backtracks
        end -= 1                      # відступити на один символ
        backtracks += 1
    return None, backtracks

In [ ]:
for page in ("<b>Python</b><i>курс</i>", "<b>Python</b> курс"):
    end, steps = greedy_star_then(page, 1, ">")
    print(f"{page!r:28} збіг: {page[:end]!r:28} відступів: {steps}")
    assert page[:end] == re.search(r"<.*>", page).group()

На першому рядку відступів **0**: рядок і так закінчується на `>`, тож `.*` одразу «влучив» в останню дужку. На другому — **5**: `.*` повертає по одному символу хвоста ` курс`, доки не натрапить на `>`. В обох випадках збіг імітації такий самий, як у справжнього `re.search` (це перевіряє `assert`), — і закінчується він на **останній** `>`.

### Приклад 5.6. Катастрофічне відступання

Відступання зазвичай дешеве. Але вкладені квантифікатори на кшталт `(a+)+` на рядку, що **майже** підходить, змушують рушій перебрати всі способи розбити текст на групи — їх кількість росте експоненційно. Заміряємо на коротких рядках (довші завісили б клітинку).

In [ ]:
import time


def time_match(pattern, text):
    start = time.perf_counter()
    result = re.fullmatch(pattern, text)
    return result, time.perf_counter() - start

In [ ]:
for n in (16, 18, 20, 22):
    text = "a" * n + "!"
    _, slow = time_match(r"(a+)+", text)
    _, fast = time_match(r"a+", text)
    print(f"n = {n}: (a+)+ — {slow:.3f} с,  a+ — {fast:.6f} с")

Обидва шаблони кажуть «відповідей немає» (у кінці `!`), але `(a+)+` на кожні +2 символи працює в 3–4 рази довше, а простий `a+` — мить. Точні секунди в тебе інші — важливе співвідношення. **Правило:** не вкладай квантифікатори один в один без потреби; у сервісах, що перевіряють текст від користувачів, такий шаблон — уразливість (ReDoS).

### Жадібний квантифікатор ≠ жадібний алгоритм

### Жадібний квантифікатор ≠ жадібний алгоритм

| | Жадібний квантифікатор `.*` | Жадібний алгоритм (урок 27) |
|---|---|---|
| Що «жадібно» | захоплює якомога більше символів | бере найкращий варіант на цьому кроці |
| Чи переглядає рішення | **так**: відступає (backtracking), якщо решта шаблону не збігається | **ніколи**: прийняте рішення остаточне |
| Гарантія | знайде збіг, якщо він існує | оптимальність лише для задач з доведенням |

Спільна лише ідея «спершу — якомога більше». І ще: `.*?` не обов'язково швидший за `.*` — нежадібний перевіряє можливість зупинки після кожного символу, і на деяких текстах це більше роботи.

### Задача 2. Побажання в лапках

Напиши `quoted(text)`: поверни список текстів, що стоять у подвійних лапках, **без самих лапок**. Подумай, який квантифікатор (або який набір) потрібен, щоб дві пари лапок не злиплися в одну, — приклади 5.2 і 5.3. Порожні лапки `""` дають порожній рядок.

Перевірка — у наступній клітинці.

In [ ]:
def quoted(text):
    # BEGIN SOLUTION
    return re.findall(r'"(.*?)"', text)
    # END SOLUTION

In [ ]:
print(quoted(journal))
assert quoted(journal) == ["без цибулі", "дзвонити в двері"]
assert quoted('лише "одне"') == ["одне"]
assert quoted('порожні "" і "повні"') == ["", "повні"]
assert quoted("без лапок") == []
print("✅ Задача 2 розв'язана")

---
## 6. Межі regex: чому не для HTML

### Приклад 6.1. Вкладені теги

In [ ]:
html = "<div><div>вкладений</div> текст</div>"
print(re.findall(r"<div>(.*?)</div>", html))
print(re.findall(r"<div>(.*)</div>", html))

Жоден варіант не дав «вміст зовнішнього `div`» правильно: нежадібний зупинився на першому `</div>` (внутрішньому), жадібний — захопив теги разом з вмістом. Regex не рахує вкладеність: для нього немає «цей `</div>` закриває той `<div>`».

Реальний HTML ще й буває незакритим, з атрибутами в довільному порядку, коментарями та скриптами. Для HTML використовують парсери, що будують дерево елементів: `html.parser` зі стандартної бібліотеки (приклад 6.2), BeautifulSoup, lxml. Regex — для простих **плоских** шаблонів: номерів, дат, часу, кодів.

### Приклад 6.2. `html.parser`: парсер рахує вкладеність

`HTMLParser` зі стандартної бібліотеки читає HTML по тегах і викликає наші методи: відкрився тег, закрився тег, трапився текст. Лічильник `depth` показує, наскільки глибоко ми всередині `div`. Так можна зібрати текст **зовнішнього** блоку — те, чого не вмів regex.

In [ ]:
from html.parser import HTMLParser


class DivText(HTMLParser):
    """Збирає текст усередині div разом з глибиною вкладеності."""

    def __init__(self):
        super().__init__()
        self.depth = 0
        self.parts = []

    def handle_starttag(self, tag, attrs):
        if tag == "div":
            self.depth += 1

    def handle_endtag(self, tag):
        if tag == "div":
            self.depth -= 1

    def handle_data(self, data):
        if self.depth:
            self.parts.append((self.depth, data))

In [ ]:
parser = DivText()
parser.feed("<div><div>вкладений</div> текст</div>")
print(parser.parts)
print("".join(text for _, text in parser.parts))

Парсер знає, що `вкладений` лежить на глибині 2, а ` текст` — на глибині 1, і що обидва належать зовнішньому `div`. Склеєний текст зовнішнього блоку — `вкладений текст`. Regex для такого не годиться: він не тримає «стек» відкритих тегів.

---
## 7. Журнал диспетчерської

### Приклад 7.1. Звіт по журналу

Зводимо функції з попередніх розділів: номери замовлень, побажання, суми з журналу поїздок і акуратний коментар.

In [ ]:
trips = """09:15 #104 Хрещатик 250 грн
10:02 #105 Поділ 180 грн
збій зв'язку
10:40 Order-106 Оболонь 320 грн"""


def journal_report(text):
    return {
        "замовлення": order_numbers(text),
        "дати (формат)": re.findall(r"\b\d{2}\.\d{2}\.\d{4}\b", text),
        "суми, грн": [int(x) for x in re.findall(r"(\d+) грн", text)],
        "рядків": len(text.splitlines()),
    }

In [ ]:
for key, value in journal_report(journal).items():
    print(f"{key:14} {value}")
print()
for key, value in journal_report(trips).items():
    print(f"{key:14} {value}")

У денному журналі — три номери й три дати **за форматом** (включно з неіснуючою `31.02.2024` — календар перевіряє лише `valid_dates` із задачі 1); сум немає. У журналі поїздок — номери `#104`, `#105`, `Order-106` і суми 250, 180, 320. Рядок `збій зв'язку` не заважає пошуку — але щоб розібрати рядки поїздок **цілком**, потрібен шаблон на весь рядок.

### Задача 3. Рядки журналу → кортежі

Кожен рядок журналу поїздок має вигляд `ГГ:ХХ #номер район сума грн` (номер може бути і `Order-номер`). Напиши `parse_trips(text)`: поверни список кортежів `(час, номер, сума)`, де номер і сума — **цілі числа**. Рядки, що не відповідають формату, пропусти.

Підказки: `^…$` разом з `re.M` (приклад 4.4), група без запам'ятовування для `#`/`Order-` (приклад 3.5), нежадібний `.*?` для району, `int()` для чисел (приклад 4.3). Перевірка — у наступній клітинці.

In [ ]:
def parse_trips(text):
    # BEGIN SOLUTION
    pattern = r"^(\d{2}:\d{2}) (?:#|Order-)(\d+) .*? (\d+) грн$"
    return [(t, int(n), int(s)) for t, n, s in re.findall(pattern, text, re.M)]
    # END SOLUTION

In [ ]:
print(parse_trips(trips))
assert parse_trips(trips) == [("09:15", 104, 250), ("10:02", 105, 180), ("10:40", 106, 320)]
assert sum(amount for _, _, amount in parse_trips(trips)) == 750
assert parse_trips("11:00 #107 Нивки Святошин 400 грн") == [("11:00", 107, 400)]
assert parse_trips("збій\n11:00 #108 Поділ сто грн") == []
print("✅ Задача 3 розв'язана")

---
## Типові помилки

1. **Шаблон без `r`** — `"\b"` стає backspace, збігу немає (розділ 2).
2. **`re.match` замість `re.search`** — `match` шукає лише з початку рядка.
3. **Звичайна група там, де потрібен весь збіг** — `findall` повертає вміст групи; для `|` пиши `(?:...)`.
4. **Жадібний `.*` між розділювачами** — захоплює все до останнього розділювача; потрібен `.*?` або точніший клас (`[^"]*`).
5. **Regex для перевірки змісту** — формат дати ≠ справжня дата; перевіряй через `datetime`.
6. **Regex для HTML** — використовуй парсер.
7. **`+?` / `*?` без «стоп-сигналу» після них** — беруть мінімум, тобто один символ або нічого (приклад 5.4).
8. **Вкладені квантифікатори** `(a+)+` — експоненційне відступання на тексті, що майже підходить (приклад 5.6).

## Далі

- Урок 27 — жадібні алгоритми й динамічне програмування: [книга](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_27/).
- Документація: [`re`](https://docs.python.org/3/library/re.html), [Regular Expression HOWTO](https://docs.python.org/3/howto/regex.html) (розділ «Greedy versus Non-Greedy»), [`datetime.strptime`](https://docs.python.org/3/library/datetime.html#datetime.datetime.strptime), [`html.parser`](https://docs.python.org/3/library/html.parser.html).